In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [ ]:
!nvidia-smi


Tue Jun  2 08:26:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving processed_for_colab.zip to processed_for_colab.zip


In [ ]:
!unzip -o processed_for_colab.zip -d data
!find data -maxdepth 3 -type f

Archive:  processed_for_colab.zip
  inflating: data/processed/dataset_stats.json  
  inflating: data/processed/label2id.json  
  inflating: data/processed/test.jsonl  
  inflating: data/processed/train.jsonl  
  inflating: data/processed/val.jsonl  
data/processed/dataset_stats.json
data/processed/train.jsonl
data/processed/val.jsonl
data/processed/test.jsonl
data/processed/label2id.json


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data/processed")

def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

train_rows = read_jsonl(DATA_DIR / "train.jsonl")
val_rows = read_jsonl(DATA_DIR / "val.jsonl")
test_rows = read_jsonl(DATA_DIR / "test.jsonl")
label2id = json.load(open(DATA_DIR / "label2id.json", encoding="utf-8"))
id2label = {v: k for k, v in label2id.items()}

print(len(train_rows), len(val_rows), len(test_rows))
print(label2id)
print(train_rows[0])

6130 335 3
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ADDR': 3, 'I-ADDR': 4, 'B-NOTE': 5, 'I-NOTE': 6}
{'id': 'synth-5feaad3f', 'text': 'Vũ Thị Mai, [PHONE], 56 Trần Hưng Đạo, Q.Hoàn Kiếm, HN', 'entities': [{'start': 0, 'end': 10, 'label': 'PER'}, {'start': 21, 'end': 54, 'label': 'ADDR'}], 'source': 'claude_synth', 'variation': 'separator_only', 'created_at': '2026-06-02T04:23:17.496940+00:00'}


In [ ]:
VALID_LABELS = {"PER", "ADDR", "NOTE"}

def is_valid_example(ex):
    text = ex["text"]
    prev_end = -1
    for ent in sorted(ex["entities"], key=lambda e: e["start"]):
        s, e, label = ent["start"], ent["end"], ent["label"]
        if not (0 <= s < e <= len(text)):
            return False
        if s < prev_end:
            return False
        if label not in VALID_LABELS:
            return False
        prev_end = e
    return True

for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    bad = [i for i, ex in enumerate(rows) if not is_valid_example(ex)]
    print(name, "bad", len(bad))

train bad 0
val bad 0
test bad 0


In [ ]:
from transformers import AutoTokenizer

# Đổi sang XLM-RoBERTa để tương thích với văn bản thô và hỗ trợ Fast Tokenizer
MODEL_NAME = "xlm-roberta-base"

# AutoTokenizer lúc này sẽ tự động gọi XLMRobertaTokenizerFast và sinh ra offset_mapping
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [ ]:
ENTITY_TO_BI = {
    "PER": ("B-PER", "I-PER"),
    "ADDR": ("B-ADDR", "I-ADDR"),
    "NOTE": ("B-NOTE", "I-NOTE"),
}

def char_label_at(token_start, token_end, entities):
    for ent in entities:
        s, e, label = ent["start"], ent["end"], ent["label"]
        if token_start >= s and token_end <= e:
            b_label, i_label = ENTITY_TO_BI[label]
            return b_label if token_start == s else i_label
    return "O"

def tokenize_and_align(ex):
    enc = tokenizer(
        ex["text"],
        truncation=True,
        max_length=256,
        return_offsets_mapping=True,
    )
    labels = []
    for start, end in enc["offset_mapping"]:
        if start == end:
            labels.append(-100)
        else:
            labels.append(label2id[char_label_at(start, end, ex["entities"])])
    enc.pop("offset_mapping")
    enc["labels"] = labels
    return enc

In [ ]:
sample = tokenize_and_align(train_rows[0])
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
for tok, lab in zip(tokens[:80], sample["labels"][:80]):
    print(tok, id2label[lab] if lab != -100 else "IGN")

<s> IGN
▁Vũ B-PER
▁Thị I-PER
▁Mai I-PER
, O
▁[ O
PHONE O
] O
, O
▁56 B-ADDR
▁Trần I-ADDR
▁Hưng I-ADDR
▁Đạo I-ADDR
, I-ADDR
▁Q I-ADDR
. I-ADDR
Ho I-ADDR
àn I-ADDR
▁Kiếm I-ADDR
, I-ADDR
▁ I-ADDR
HN I-ADDR
</s> IGN


In [ ]:
from datasets import Dataset, DatasetDict

raw_ds = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
    "test": Dataset.from_list(test_rows),
})

# Dùng vòng lặp để duyệt qua từng split (train, validation, test)
# và lấy đúng column_names của split đó để remove
tokenized_ds = DatasetDict({
    split: dataset.map(tokenize_and_align, remove_columns=dataset.column_names)
    for split, dataset in raw_ds.items()
})

Map:   0%|          | 0/6130 [00:00<?, ? examples/s]

Map:   0%|          | 0/335 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_ds)
print(tokenized_ds["train"][0].keys())

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6130
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 335
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3
    })
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


## M6: tiny smoke training

In [35]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

tiny_train = tokenized_ds["train"].select(range(64))
tiny_val = tokenized_ds["validation"].select(range(min(64, len(tokenized_ds["validation"]))))

args = TrainingArguments(
    output_dir="runs/xlmr_ner_tiny",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tiny_train,
    eval_dataset=tiny_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,1.711150


TrainOutput(global_step=8, training_loss=1.8353822231292725, metrics={'train_runtime': 2.1176, 'train_samples_per_second': 30.223, 'train_steps_per_second': 3.778, 'total_flos': 1322874628416.0, 'train_loss': 1.8353822231292725, 'epoch': 1.0})

In [36]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        cur_preds = []
        cur_labels = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            cur_preds.append(id2label[int(pred_id)])
            cur_labels.append(id2label[int(label_id)])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    result = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": result["overall_precision"],
        "recall": result["overall_recall"],
        "f1": result["overall_f1"],
        "accuracy": result["overall_accuracy"],
    }

In [38]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
args = TrainingArguments(
    output_dir="runs/xlmr_ner_baseline",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print(val_metrics)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.054862,0.039355,0.920428,0.928144,0.924270,0.983815
2,0.052417,0.029525,0.946365,0.950898,0.948626,0.987861


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.054862,0.039355,0.920428,0.928144,0.924270,0.983815
2,0.052417,0.029525,0.946365,0.950898,0.948626,0.987861
3,0.035482,0.026256,0.948931,0.956886,0.952892,0.988423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': 0.026256341487169266, 'eval_precision': 0.9489311163895487, 'eval_recall': 0.9568862275449102, 'eval_f1': 0.9528920691711389, 'eval_accuracy': 0.9884230639541418, 'eval_runtime': 1.1359, 'eval_samples_per_second': 294.926, 'eval_steps_per_second': 36.976, 'epoch': 3.0}


In [41]:
test_metrics = trainer.evaluate(tokenized_ds["test"])
print(test_metrics)

{'eval_loss': 0.0012318241642788053, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0, 'eval_accuracy': 1.0, 'eval_runtime': 0.037, 'eval_samples_per_second': 81.068, 'eval_steps_per_second': 27.023, 'epoch': 3.0}


In [49]:
import torch

def predict_token_labels(text):
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(trainer.model.device) for k, v in enc.items()}

    with torch.no_grad():
        logits = trainer.model(**enc).logits[0].detach().cpu().numpy()

    pred_ids = logits.argmax(axis=-1).tolist()
    input_ids = enc["input_ids"][0].detach().cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    rows = []
    for i, (tok, off, pred_id) in enumerate(zip(tokens, offsets, pred_ids)):
        if off[0] == off[1]:
            continue
        rows.append((tok, off, id2label[pred_id]))
    return rows

samples = [
    # =========================
    # TC-P — Parsing
    # =========================
    "Nguyễn Văn A, 0912345678, 45 Lê Lợi Q1 HCM, giao buổi sáng",
    "sdt: 0987654321 - địa chỉ: 12 Trần Hưng Đạo, Đống Đa, Hà Nội",
    "chị Mai ơi giao cho mình nha, đang ở 88 Nguyễn Du",
    "Tên: Trần Bích Ngọc | ĐT: 0901 234 567 | Địa chỉ: Căn hộ 12B, Chung cư Sunrise, 90 Võ Văn Ngân, Thủ Đức | Ghi chú: gọi trước 30p",
    "order mới nè: 5 áo size M màu đen, ship cho Hùng, 0911222333, 22 Hai Bà Trưng",
    "không có địa chỉ, tên: Lan, sdt: 0999888777",
    "",
    "+84 912 345 678",
    'Lê Văn Đức, 0812456789, 50 Đinh Tiên Hoàng Q.Bình Thạnh - nhớ ghi "hàng dễ vỡ" lên kiện',
    """Nguyễn Thị Hoa
0934567890
15 Lý Thường Kiệt, P.14, Q.10""",

    # =========================
    # TC-PH — Phone Validation
    # =========================
    "0912 345 678",
    "0912-345-678",
    "84912345678",
    "84912345678",
    "(0912) 345.678",
    "091234567",
    "09123456789",
    "0123456789",
    "nhà số 0987654321, giao buổi chiều",
    "mã đơn: 0912345678",
    "Lan 0901234567, liên hệ shop 0987654321",
    "0901234567 hoặc 0987654321 đều được",
    "Không có số nào trong text",

    # =========================
    # TC-A — Address Autocorrect
    # =========================
    "45 Le Loi, Q1, TP.HCM",
    "Nguyen Hue, Ha Noi",
    "45 Lê Lợi, Q1",
    "HN, Đống Đa, Hàng Bông 18",
    "123 đường số 5, P. bình hưng hòa, Bình Tân",
    "Hàng Bông 18, Đống Đa",
    "địa chỉ không tồn tại @#$%",
    "Quận 3, HCM",

    # =========================
    # TC-E — Edge Cases
    # =========================
    "giao cho mình trước 10h sáng nhé, 45 Lê Lợi Q1",
    "Text từ Facebook :v :3 😀 😆 ship tới 22 Nguyễn Huệ Q1 nhé",
    "Số nhà TT08, KĐT Vinhomes Smart City, Nam Từ Liêm, Hà Nội",
    "Trường THPT Chu Văn An, Tây Hồ, Hà Nội",
    "lấy hàng ở 12 Lý Thái Tổ, giao tới 88 Nguyễn Du",
    "Lan ơi ship cho mình 1 cái váy size S nha, mình ở 22 Ngô Quyền, phone mình 0901234567",

    # =========================
    # TC-R — API / Security
    # =========================
    "<script>alert('xss')</script> ship tới 45 Lê Lợi Q1",
    "😀😀😀 giao giúp mình tới 22 Nguyễn Huệ Q1 nha ❤️",
    "🔥🔥🔥 Nguyễn Văn A - 0912345678 - 45 Lê Lợi Q1",
]
for text in samples:
    print("\\nTEXT:", text)
    for row in predict_token_labels(text):
        print(row)

\nTEXT: Nguyễn Văn A, 0912345678, 45 Lê Lợi Q1 HCM, giao buổi sáng
('▁Nguyễn', [0, 6], 'B-PER')
('▁Văn', [7, 10], 'I-PER')
('▁A', [11, 12], 'I-PER')
(',', [12, 13], 'O')
('▁09', [14, 16], 'O')
('12', [16, 18], 'O')
('345', [18, 21], 'O')
('678', [21, 24], 'O')
(',', [24, 25], 'O')
('▁45', [26, 28], 'B-ADDR')
('▁Lê', [29, 31], 'I-ADDR')
('▁Lợi', [32, 35], 'I-ADDR')
('▁Q', [36, 37], 'I-ADDR')
('1', [37, 38], 'I-ADDR')
('▁HCM', [39, 42], 'I-ADDR')
(',', [42, 43], 'O')
('▁giao', [44, 48], 'B-NOTE')
('▁buổi', [49, 53], 'I-NOTE')
('▁sáng', [54, 58], 'I-NOTE')
\nTEXT: sdt: 0987654321 - địa chỉ: 12 Trần Hưng Đạo, Đống Đa, Hà Nội
('▁s', [0, 1], 'O')
('dt', [1, 3], 'O')
(':', [3, 4], 'O')
('▁09', [5, 7], 'O')
('8', [7, 8], 'O')
('765', [8, 11], 'O')
('43', [11, 13], 'O')
('21', [13, 15], 'O')
('▁-', [16, 17], 'O')
('▁địa', [18, 21], 'O')
('▁chỉ', [22, 25], 'O')
(':', [25, 26], 'O')
('▁12', [27, 29], 'B-ADDR')
('▁Trần', [30, 34], 'I-ADDR')
('▁Hưng', [35, 39], 'I-ADDR')
('▁Đạo', [40, 43], 'I-ADDR'

In [44]:
EXPORT_DIR = "ner_xlmr_clipboard"

trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

import json
with open(f"{EXPORT_DIR}/label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

!zip -r ner_xlmr_clipboard.zip ner_xlmr_clipboard

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

updating: ner_xlmr_clipboard/ (stored 0%)
updating: ner_xlmr_clipboard/training_args.bin (deflated 53%)
updating: ner_xlmr_clipboard/tokenizer.json (deflated 77%)
updating: ner_xlmr_clipboard/config.json (deflated 54%)
updating: ner_xlmr_clipboard/model.safetensors (deflated 26%)
updating: ner_xlmr_clipboard/tokenizer_config.json (deflated 47%)
updating: ner_xlmr_clipboard/label2id.json (deflated 45%)


In [50]:
from google.colab import files
files.download("ner_xlmr_clipboard.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>